# Qualitative Analysis

Survival curves, KM comparisons, σ stability, and dataset summaries.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from lifelines import KaplanMeierFitter

from src.utils import DATASETS, make_splits
from src.fsa import predict_log_time, fit_sigma, survival_lognormal, predicted_median

plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = sns.color_palette('tab10')
EPS = 1e-8
DATASET = 'gbsg'   # change to explore others

## 1. Censoring Summary

In [ ]:
import pandas as pd

rows = []
for name, loader in DATASETS.items():
    X, T, Delta = loader()
    rows.append({
        'dataset': name,
        'n': len(T),
        'censoring_rate': f"{(1 - Delta.mean()):.2%}",
        'median_followup': f"{np.median(T):.1f}",
    })
pd.DataFrame(rows).set_index('dataset')

## 2. Survival Curves vs KM (by Risk Quartile)

In [ ]:
X, T, Delta = DATASETS[DATASET]()
tr_idx, te_idx = make_splits(len(T), n_splits=1)[0]
X_tr, T_tr, D_tr = X[tr_idx], T[tr_idx], Delta[tr_idx]
X_te, T_te, D_te = X[te_idx], T[te_idx], Delta[te_idx]

uncensored = D_tr == 1
log_T_unc = np.log(np.clip(T_tr[uncensored], EPS, None))
X_all = np.vstack([X_tr, X_te])
mu_all = predict_log_time(X_tr[uncensored], log_T_unc, X_all)
mu_tr, mu_te = mu_all[:len(X_tr)], mu_all[len(X_tr):]
sigma = fit_sigma(T_tr, D_tr, mu_tr)

t_grid = np.linspace(np.percentile(T_tr, 1), np.percentile(T_tr, 99), 300)
S = survival_lognormal(t_grid, mu_te, sigma)
med = predicted_median(S, t_grid)

# Risk quartiles: lower median = higher risk
quartiles = np.percentile(med[np.isfinite(med)], [25, 50, 75])
labels = ['Q1 (high risk)', 'Q2', 'Q3', 'Q4 (low risk)']
bins = np.digitize(med, quartiles)

fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
kmf = KaplanMeierFitter()
for q, (ax, label) in enumerate(zip(axes, labels)):
    mask = bins == q
    if mask.sum() == 0:
        continue
    # Predicted mean curve
    ax.plot(t_grid, S[mask].mean(0), color=PALETTE[q], lw=2, label='Predicted')
    # KM estimate
    kmf.fit(T_te[mask], event_observed=D_te[mask])
    kmf.plot_survival_function(ax=ax, ci_show=True, color=PALETTE[q],
                               linestyle='--', label='KM', linewidth=1.5)
    ax.set_title(label, fontsize=11)
    ax.set_xlabel('Time')
    ax.legend(fontsize=9)
axes[0].set_ylabel('S(t)')
fig.suptitle(f'Predicted survival vs KM — {DATASET}', fontsize=13)
plt.tight_layout()
plt.savefig(f'../figures/{DATASET}_km_quartiles.pdf', dpi=300, bbox_inches='tight')
plt.show()

## 3. Individual Survival Curves

In [ ]:
n_subjects = 6
# Pick subjects spread across the risk spectrum
idx = np.linspace(0, len(med) - 1, n_subjects, dtype=int)[np.argsort(np.argsort(med))[:n_subjects]]
idx = np.argsort(med)[[0, len(med)//5, 2*len(med)//5, 3*len(med)//5, 4*len(med)//5, -1]]

fig, ax = plt.subplots(figsize=(8, 5))
for i, j in enumerate(idx):
    label = f'Subject {j} (med={med[j]:.0f})'
    ax.plot(t_grid, S[j], color=PALETTE[i], lw=2, label=label)
ax.axhline(0.5, color='grey', linestyle=':', lw=1)
ax.set_xlabel('Time'); ax.set_ylabel('S(t | x)')
ax.set_title(f'Individual survival curves — {DATASET}')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f'../figures/{DATASET}_individual_curves.pdf', dpi=300, bbox_inches='tight')
plt.show()

## 4. σ Stability Across Datasets and Splits

In [ ]:
sigma_results = {}
for ds_name, loader in DATASETS.items():
    X, T, Delta = loader()
    sigmas = []
    for tr_idx, te_idx in make_splits(len(T), n_splits=10):
        X_tr, T_tr, D_tr = X[tr_idx], T[tr_idx], Delta[tr_idx]
        unc = D_tr == 1
        log_T_unc = np.log(np.clip(T_tr[unc], EPS, None))
        mu_tr = predict_log_time(X_tr[unc], log_T_unc, X_tr)
        sigmas.append(fit_sigma(T_tr, D_tr, mu_tr))
    sigma_results[ds_name] = sigmas

fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot(sigma_results.values(), labels=sigma_results.keys(), patch_artist=True)
ax.axhline(1.0, color='grey', linestyle='--', lw=1, label='σ = 1')
ax.set_ylabel('σ'); ax.set_title('Fitted σ across datasets and splits')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/sigma_stability.pdf', dpi=300, bbox_inches='tight')
plt.show()